# 📊 Speech Emotion Recognition — Exploratory Data Analysis (EDA)

## 🎯 Objectif
Analyser en profondeur le dataset **RAVDESS** (Ryerson Audio-Visual Database of Emotional Speech and Song) pour comprendre sa structure, ses caractéristiques acoustiques et la distribution des émotions avant le preprocessing et l’entraînement.

## 📁 Structure du Dataset RAVDESS
- **24 acteurs** (12 hommes, 12 femmes)
- **8 émotions** : neutral, calm, happy, sad, angry, fearful, disgust, surprise
- **Fichiers audio** : format WAV, 48kHz, 16-bit
- **Convention de nommage** : `Modality-VocalChannel-Emotion-Intensity-Statement-Repetition-Actor.wav`

## 📚 Références
- Livingstone & Russo (2018). *The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS)*
- Zhao, J., et al. (2019). *Speech emotion recognition using deep 1D & 2D CNN LSTM networks*

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import IPython.display as ipd
from collections import Counter

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully.')

## 1. Chargement et Parsing du Dataset

In [ ]:
DATA_PATH = 'RAVDESS'

# Mapping des identifiants RAVDESS
EMOTION_MAP = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprise'
}

INTENSITY_MAP = {'01': 'normal', '02': 'strong'}
STATEMENT_MAP = {'01': 'Kids are talking by the door', '02': 'Dogs are sitting by the door'}

data = []

for actor_folder in sorted(os.listdir(DATA_PATH)):
    actor_path = os.path.join(DATA_PATH, actor_folder)
    if not os.path.isdir(actor_path) or actor_folder == 'audio_speech_actors_01-24':
        continue  # skip nested duplicate folder
    for file in sorted(os.listdir(actor_path)):
        if not file.endswith('.wav'):
            continue
        parts = file.replace('.wav', '').split('-')
        # Modality-VocalChannel-Emotion-Intensity-Statement-Repetition-Actor
        actor_id = int(parts[6])
        gender = 'male' if actor_id % 2 == 1 else 'female'
        data.append({
            'path': os.path.join(actor_path, file),
            'filename': file,
            'modality': parts[0],
            'vocal_channel': parts[1],
            'emotion': EMOTION_MAP.get(parts[2], 'unknown'),
            'emotion_id': parts[2],
            'intensity': INTENSITY_MAP.get(parts[3], 'unknown'),
            'statement': STATEMENT_MAP.get(parts[4], 'unknown'),
            'repetition': int(parts[5]),
            'actor_id': actor_id,
            'gender': gender
        })

df = pd.DataFrame(data)
print(f'Total fichiers audio : {len(df)}')
print(f'Acteurs uniques : {df["actor_id"].nunique()}')
print(f'\u00c9motions : {df["emotion"].unique().tolist()}')
df.head(10)

## 2. Analyse de la Distribution des Émotions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution globale des émotions
emotion_order = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprise']
colors = sns.color_palette('Set2', len(emotion_order))

sns.countplot(data=df, x='emotion', order=emotion_order, palette=colors, ax=axes[0])
axes[0].set_title('Distribution globale des \u00e9motions')
axes[0].set_xlabel('\u00c9motion')
axes[0].set_ylabel('Nombre de fichiers')
axes[0].tick_params(axis='x', rotation=45)

# Ajout des valeurs sur les barres
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)

# Distribution par genre
sns.countplot(data=df, x='emotion', hue='gender', order=emotion_order, palette='coolwarm', ax=axes[1])
axes[1].set_title('Distribution des \u00e9motions par genre')
axes[1].set_xlabel('\u00c9motion')
axes[1].set_ylabel('Nombre de fichiers')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Genre')

plt.tight_layout()
plt.show()

print('\nR\u00e9partition par \u00e9motion :')
print(df['emotion'].value_counts().to_string())

In [ ]:
# Distribution par intensité et émotion
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=df, x='emotion', hue='intensity', order=emotion_order, palette='viridis', ax=axes[0])
axes[0].set_title('Distribution des \u00e9motions par intensit\u00e9')
axes[0].set_xlabel('\u00c9motion')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Intensit\u00e9')

# Acteurs par genre
actor_gender = df.drop_duplicates('actor_id')[['actor_id', 'gender']]
sns.countplot(data=actor_gender, x='gender', palette='coolwarm', ax=axes[1])
axes[1].set_title('R\u00e9partition des acteurs par genre')
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Nombre d\'acteurs')

plt.tight_layout()
plt.show()

## 3. Analyse des Durées Audio

In [ ]:
# Calcul des durées
durations = []
sample_rates = []

for idx, row in df.iterrows():
    y, sr = librosa.load(row['path'], sr=None)  # sr=None pour garder le SR original
    duration = len(y) / sr
    durations.append(duration)
    sample_rates.append(sr)

df['duration'] = durations
df['sample_rate'] = sample_rates

print(f'Dur\u00e9e moyenne : {df["duration"].mean():.2f}s')
print(f'Dur\u00e9e min : {df["duration"].min():.2f}s')
print(f'Dur\u00e9e max : {df["duration"].max():.2f}s')
print(f'\u00c9cart-type : {df["duration"].std():.2f}s')
print(f'Sample rates uniques : {df["sample_rate"].unique()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogramme des durées
axes[0].hist(df['duration'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['duration'].mean(), color='red', linestyle='--', label=f'Moyenne: {df["duration"].mean():.2f}s')
axes[0].set_title('Distribution des dur\u00e9es audio')
axes[0].set_xlabel('Dur\u00e9e (secondes)')
axes[0].set_ylabel('Fr\u00e9quence')
axes[0].legend()

# Durées par émotion (boxplot)
sns.boxplot(data=df, x='emotion', y='duration', order=emotion_order, palette='Set2', ax=axes[1])
axes[1].set_title('Dur\u00e9e par \u00e9motion')
axes[1].set_xlabel('\u00c9motion')
axes[1].set_ylabel('Dur\u00e9e (secondes)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Durée par acteur et genre
fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=df, x='actor_id', y='duration', hue='gender', palette='coolwarm', ax=ax)
ax.set_title('Dur\u00e9e audio par acteur')
ax.set_xlabel('Acteur ID')
ax.set_ylabel('Dur\u00e9e (secondes)')
ax.legend(title='Genre')
plt.tight_layout()
plt.show()

## 4. Visualisation des Signaux Audio

In [ ]:
# Visualiser un signal par émotion
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
axes = axes.flatten()

for i, emotion in enumerate(emotion_order):
    sample = df[df['emotion'] == emotion].iloc[0]
    y, sr = librosa.load(sample['path'], sr=22050)
    librosa.display.waveshow(y, sr=sr, ax=axes[i], color=colors[i])
    axes[i].set_title(f'{emotion.upper()} (Actor {sample["actor_id"]}, {sample["gender"]})')
    axes[i].set_xlabel('Temps (s)')
    axes[i].set_ylabel('Amplitude')

plt.suptitle('Formes d\'onde par \u00e9motion', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 5. Analyse Spectrale

In [ ]:
# Mel Spectrogrammes par émotion
fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.flatten()

for i, emotion in enumerate(emotion_order):
    sample = df[df['emotion'] == emotion].iloc[0]
    y, sr = librosa.load(sample['path'], sr=22050)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[i])
    axes[i].set_title(f'{emotion.upper()}')
    fig.colorbar(img, ax=axes[i], format='%+2.0f dB')

plt.suptitle('Mel Spectrogrammes par \u00e9motion', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# MFCCs par émotion
fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.flatten()

for i, emotion in enumerate(emotion_order):
    sample = df[df['emotion'] == emotion].iloc[0]
    y, sr = librosa.load(sample['path'], sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    img = librosa.display.specshow(mfcc, sr=sr, x_axis='time', ax=axes[i])
    axes[i].set_title(f'{emotion.upper()}')
    fig.colorbar(img, ax=axes[i])

plt.suptitle('MFCCs par \u00e9motion', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 6. Analyse Statistique des Features Acoustiques

In [ ]:
# Extraire des features statistiques pour chaque fichier
stats = []

for idx, row in df.iterrows():
    y, sr = librosa.load(row['path'], sr=22050, duration=3)
    
    # Features de base
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    rms = librosa.feature.rms(y=y)[0]
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    f0, voiced_flag, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
    f0_clean = f0[~np.isnan(f0)] if f0 is not None else np.array([0])
    
    stats.append({
        'emotion': row['emotion'],
        'gender': row['gender'],
        'zcr_mean': zcr.mean(),
        'zcr_std': zcr.std(),
        'rms_mean': rms.mean(),
        'rms_std': rms.std(),
        'centroid_mean': spectral_centroid.mean(),
        'centroid_std': spectral_centroid.std(),
        'bandwidth_mean': spectral_bandwidth.mean(),
        'rolloff_mean': spectral_rolloff.mean(),
        'f0_mean': f0_clean.mean() if len(f0_clean) > 0 else 0,
        'f0_std': f0_clean.std() if len(f0_clean) > 0 else 0,
    })

df_stats = pd.DataFrame(stats)
print('Features acoustiques extraites pour', len(df_stats), 'fichiers')
df_stats.head()

In [ ]:
# Comparaison des features par émotion
features_to_plot = ['zcr_mean', 'rms_mean', 'centroid_mean', 'f0_mean', 'bandwidth_mean', 'rolloff_mean']
titles = ['Zero Crossing Rate', 'RMS Energy', 'Spectral Centroid', 'Pitch (F0)', 'Spectral Bandwidth', 'Spectral Rolloff']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (feat, title) in enumerate(zip(features_to_plot, titles)):
    sns.boxplot(data=df_stats, x='emotion', y=feat, order=emotion_order, palette='Set2', ax=axes[i])
    axes[i].set_title(title)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('')

plt.suptitle('Caract\u00e9ristiques acoustiques par \u00e9motion', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison Homme vs Femme par feature
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (feat, title) in enumerate(zip(features_to_plot, titles)):
    sns.boxplot(data=df_stats, x='emotion', y=feat, hue='gender', order=emotion_order, palette='coolwarm', ax=axes[i])
    axes[i].set_title(title)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('')
    if i > 0:
        axes[i].get_legend().remove()

plt.suptitle('Caract\u00e9ristiques acoustiques par \u00e9motion et genre', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 7. Matrice de Corrélation des Features

In [ ]:
corr_cols = ['zcr_mean', 'zcr_std', 'rms_mean', 'rms_std', 'centroid_mean', 
             'centroid_std', 'bandwidth_mean', 'rolloff_mean', 'f0_mean', 'f0_std']

plt.figure(figsize=(12, 10))
corr = df_stats[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Matrice de corr\u00e9lation des features acoustiques')
plt.tight_layout()
plt.show()

## 8. Écoute d'Échantillons Audio

In [ ]:
# Écouter un exemple par émotion
for emotion in emotion_order:
    sample = df[df['emotion'] == emotion].iloc[0]
    print(f'\n\u00c9motion: {emotion.upper()} | Actor: {sample["actor_id"]} | Genre: {sample["gender"]}')
    display(ipd.Audio(sample['path']))

## 9. Résumé et Conclusions de l'EDA

### Observations clés :
1. **Dataset équilibré** : Toutes les émotions (sauf neutral) ont le même nombre d'échantillons
2. **Équilibre de genre** : 12 acteurs masculins et 12 féminins
3. **Durées variables** : Les durées varient entre ~3-6s selon l'émotion et l'acteur
4. **Différences acoustiques** :
   - Les émotions `angry` et `happy` montrent une énergie (RMS) et un pitch (F0) plus élevés
   - `sad` et `neutral` ont des valeurs de ZCR et centroid plus basses
   - `fearful` montre une variabilité élevée dans les mesures spectrales
5. **Corrélation forte** entre centroid, bandwidth et rolloff → redondance potentielle

### Implications pour le modèle :
- Les MFCCs capturent bien les différences inter-émotions
- Les features temporelles (ZCR, RMS) apportent une information complémentaire
- Les modèles pré-entraînés (Wav2Vec2, HuBERT) devraient mieux capturer les nuances fines

In [ ]:
# Sauvegarder le DataFrame enrichi pour le Preprocessing
df.to_csv('eda_dataframe.csv', index=False)
print('DataFrame sauvegard\u00e9 dans eda_dataframe.csv')
print(f'\nR\u00e9sum\u00e9 final :')
print(f'  - Fichiers audio : {len(df)}')
print(f'  - \u00c9motions : {df["emotion"].nunique()}')
print(f'  - Acteurs : {df["actor_id"].nunique()}')
print(f'  - Dur\u00e9e moyenne : {df["duration"].mean():.2f}s')